# GALVAT Trial 8: Roboflow-only dataset rebuild

This notebook starts from the latest Trial 7 training dataset zip, removes Microsoft COCO and Open Images examples, keeps Roboflow examples and labels, then lets you add new Roboflow datasets for missing classes.

Primary objective: build a cleaner Trial 8 dataset for the GALVAT weighted metric. mAP remains secondary.

Important ontology change: `bathroom_sign` is replaced by `mens_bathroom_sign` and `womens_bathroom_sign`. Do not import generic bathroom-sign datasets unless you can map the labels to one of those two classes.


In [ ]:
# Cell 1: Setup
!pip -q install pyyaml pandas pillow tqdm matplotlib seaborn

from google.colab import drive
drive.mount('/content/drive')

import datetime
import hashlib
import json
import math
import os
import random
import re
import shutil
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import yaml
from PIL import Image, ImageOps
from tqdm.auto import tqdm

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


In [ ]:
# Cell 2: Paths and Trial 8 ontology
# Change SOURCE_ZIP only if you have a newer dataset checkpoint.

SOURCE_ZIP = Path('/content/drive/MyDrive/galvat_trial7_training_datasets/galvat_trial7_training_dataset_640_2026-06-11_17-49-31.zip')

STAMP = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
WORK_ROOT = Path('/content/galvat_trial8_roboflow_only_working')
RESTORE_ROOT = WORK_ROOT / 'restore'
WORK_DS = WORK_ROOT / 'dataset'
WORK_META = WORK_ROOT / 'metadata'
ROBOSRC_ROOT = Path('/content/roboflow_sources_trial8')
DRIVE_OUT = Path('/content/drive/MyDrive/galvat_trial8_dataset_checkpoints')

for p in [RESTORE_ROOT, WORK_DS, WORK_META, ROBOSRC_ROOT, DRIVE_OUT]:
    p.mkdir(parents=True, exist_ok=True)

FINAL_NAMES = [
    'tupperware', 'lemon', 'jug', 'canned_good',
    'fork', 'knife', 'spoon',
    'drinking_glass', 'wine_glass', 'coffee_mug',
    'salt_pepper_shaker', 'measuring_cup', 'medicine_bottle',
    'pot', 'pan',
    'cellphone', 'wallet', 'purse', 'cane', 'keys',
    'toilet', 'urinal', 'paper_towel_dispenser', 'soap_dispenser', 'trash_can',
    'light_switch', 'electrical_outlet', 'railing',
    'tv_remote', 'writing_device', 'eyeglasses', 'hearing_aid',
    'toothbrush', 'toothpaste', 'deodorant', 'razor',
    'step_curb', 'exit_sign', 'mens_bathroom_sign', 'womens_bathroom_sign',
    'thermostat', 'door', 'window', 'chair', 'table',
]
final_id = {name: i for i, name in enumerate(FINAL_NAMES)}

TARGET_IMAGES = {
    'paper_towel_dispenser': 1500, 'hearing_aid': 1500,
    'mens_bathroom_sign': 750, 'womens_bathroom_sign': 750,
    'step_curb': 1500, 'exit_sign': 1500, 'railing': 1500, 'urinal': 1500,
    'light_switch': 1500, 'electrical_outlet': 1500, 'door': 1500,
    'tupperware': 1000, 'drinking_glass': 1000, 'razor': 1000, 'thermostat': 1000,
    'measuring_cup': 1000, 'salt_pepper_shaker': 1000, 'wallet': 1000,
    'soap_dispenser': 1000, 'pan': 1000, 'jug': 1000, 'toothpaste': 1000,
    'writing_device': 1000, 'medicine_bottle': 1000, 'toothbrush': 1000,
    'lemon': 1000, 'trash_can': 1000, 'deodorant': 1000, 'keys': 1000,
    'canned_good': 1000, 'pot': 1000, 'cane': 1000, 'coffee_mug': 1000,
    'wine_glass': 1000, 'eyeglasses': 1000, 'purse': 1000, 'toilet': 1000,
    'fork': 1000, 'knife': 1000, 'spoon': 1000, 'cellphone': 1000,
    'tv_remote': 1000, 'chair': 1000, 'table': 1000, 'window': 1000,
}

assert SOURCE_ZIP.exists(), f'Missing SOURCE_ZIP: {SOURCE_ZIP}'
print('Trial 8 class count:', len(FINAL_NAMES))
print('Source zip:', SOURCE_ZIP)
print('Working dataset:', WORK_DS)


In [ ]:
# Cell 3: Restore Trial 7 dataset zip into /content
# This does not modify the Drive zip.

if any(WORK_DS.glob('*')):
    print('Working dataset already exists. Delete WORK_ROOT manually if you want a clean rebuild:', WORK_ROOT)
else:
    print('Unzipping:', SOURCE_ZIP)
    with zipfile.ZipFile(SOURCE_ZIP, 'r') as z:
        z.extractall(RESTORE_ROOT)

    data_yaml_candidates = sorted(RESTORE_ROOT.rglob('data.yaml'))
    assert data_yaml_candidates, f'No data.yaml found under {RESTORE_ROOT}'
    BASE_DATA_YAML = data_yaml_candidates[0]
    BASE_DS = BASE_DATA_YAML.parent
    print('Base dataset:', BASE_DS)

    shutil.copytree(BASE_DS, WORK_DS, dirs_exist_ok=True)
    print('Copied to working dataset:', WORK_DS)


In [ ]:
# Cell 4: Rewrite labels/data.yaml into Trial 8 ontology
# This remaps old Trial 7 ids to the Trial 8 class ids and drops any class
# not present in FINAL_NAMES. Old bathroom_sign labels are not auto-split.

def load_yolo_names(data_yaml_path):
    with open(data_yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    names = data.get('names')
    if isinstance(names, dict):
        names = [names[i] if i in names else names[str(i)] for i in range(len(names))]
    return data, list(names)

def write_trial8_data_yaml():
    data_yaml_path = WORK_DS / 'data.yaml'
    data, old_names = load_yolo_names(data_yaml_path)
    old_to_new = {}
    dropped_old = []
    for old_id, old_name in enumerate(old_names):
        name = 'door' if old_name == 'door_frame' else old_name
        if name == 'bathroom_sign':
            dropped_old.append({'old_id': old_id, 'old_name': old_name, 'reason': 'bathroom_sign_must_be_split_manually'})
            continue
        if name in final_id:
            old_to_new[old_id] = final_id[name]
        else:
            dropped_old.append({'old_id': old_id, 'old_name': old_name, 'reason': 'not_in_trial8_ontology'})

    bad_rows = []
    remapped_rows = []
    for split in ['train', 'valid', 'test']:
        lbl_dir = WORK_DS / split / 'labels'
        if not lbl_dir.exists():
            continue
        for lbl_path in tqdm(sorted(lbl_dir.glob('*.txt')), desc=f'Remapping {split} labels'):
            out_lines = []
            for line_no, line in enumerate(lbl_path.read_text().splitlines(), start=1):
                if not line.strip():
                    continue
                parts = line.split()
                if len(parts) != 5:
                    bad_rows.append({'split': split, 'file': lbl_path.name, 'line': line_no, 'reason': 'bad_column_count', 'raw': line})
                    continue
                try:
                    old_id = int(float(parts[0]))
                    vals = [float(x) for x in parts[1:]]
                except Exception:
                    bad_rows.append({'split': split, 'file': lbl_path.name, 'line': line_no, 'reason': 'parse_error', 'raw': line})
                    continue
                if old_id not in old_to_new:
                    continue
                x, y, w, h = vals
                if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                    bad_rows.append({'split': split, 'file': lbl_path.name, 'line': line_no, 'reason': 'coord_out_of_range', 'raw': line})
                    continue
                new_id = old_to_new[old_id]
                out_lines.append(f'{new_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}')
                remapped_rows.append({'split': split, 'old_class': old_names[old_id], 'new_class': FINAL_NAMES[new_id]})
            lbl_path.write_text('\n'.join(out_lines) + ('\n' if out_lines else ''))

    data['path'] = str(WORK_DS)
    data['train'] = 'train/images'
    data['val'] = 'valid/images'
    data['test'] = 'test/images'
    data['names'] = {i: name for i, name in enumerate(FINAL_NAMES)}
    data['nc'] = len(FINAL_NAMES)
    with open(data_yaml_path, 'w') as f:
        yaml.safe_dump(data, f, sort_keys=False)

    pd.DataFrame(dropped_old).to_csv(WORK_META / 'trial8_ontology_dropped_old_classes.csv', index=False)
    pd.DataFrame(bad_rows).to_csv(WORK_META / 'trial8_initial_remap_bad_rows.csv', index=False)
    pd.DataFrame(remapped_rows).to_csv(WORK_META / 'trial8_initial_remap_manifest.csv', index=False)
    print('Updated data.yaml:', data_yaml_path)
    print('Dropped old classes:', dropped_old)
    print('Bad rows during remap:', len(bad_rows))

write_trial8_data_yaml()


In [ ]:
# Cell 5: Remove Microsoft COCO and Open Images examples
# Roboflow images usually start with rf_/rf2_ or contain .rf. in the file name.
# COCO/Open Images images are removed by file-name provenance.

def infer_source_from_name(name):
    s = str(name).lower()
    if s.startswith('coco_') or '_coco_' in s or '/coco_' in s:
        return 'coco'
    if s.startswith('oi_') or '_oi_' in s or '/oi_' in s:
        return 'openimages'
    if s.startswith(('rf_', 'rf2_', 'rf8_')) or '.rf.' in s or '_jpg.rf.' in s or '_png.rf.' in s:
        return 'roboflow'
    return 'unknown'

REMOVE_SOURCES = {'coco', 'openimages'}
REMOVE_UNKNOWN = False  # Keep this False unless you inspect unknown examples first.

removed_rows = []
kept_rows = []
for split in ['train', 'valid', 'test']:
    img_dir = WORK_DS / split / 'images'
    lbl_dir = WORK_DS / split / 'labels'
    if not img_dir.exists():
        continue
    for img_path in tqdm(sorted([p for p in img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS]), desc=f'Filtering {split} provenance'):
        src = infer_source_from_name(img_path.name)
        should_remove = src in REMOVE_SOURCES or (REMOVE_UNKNOWN and src == 'unknown')
        lbl_path = lbl_dir / f'{img_path.stem}.txt'
        row = {'split': split, 'image': img_path.name, 'label': lbl_path.name, 'source': src}
        if should_remove:
            img_path.unlink(missing_ok=True)
            lbl_path.unlink(missing_ok=True)
            removed_rows.append(row)
        else:
            kept_rows.append(row)

removed_df = pd.DataFrame(removed_rows)
kept_df = pd.DataFrame(kept_rows)
removed_df.to_csv(WORK_META / 'removed_coco_openimages_manifest.csv', index=False)
kept_df.to_csv(WORK_META / 'kept_after_source_filter_manifest.csv', index=False)

print('Removed images:', len(removed_df))
print(removed_df['source'].value_counts() if len(removed_df) else pd.Series(dtype=int))
print('Kept images:', len(kept_df))
print(kept_df['source'].value_counts() if len(kept_df) else pd.Series(dtype=int))


In [ ]:
# Cell 6: Audit current Roboflow-only working dataset

def label_file_for_image(img_path):
    return Path(str(img_path).replace('/images/', '/labels/')).with_suffix('.txt')

def audit_dataset(dataset_root=WORK_DS):
    data, names = load_yolo_names(dataset_root / 'data.yaml')
    split_rows = []
    class_rows = []
    box_counts = Counter()
    image_counts = Counter()
    source_counts = Counter()
    empty_counts = Counter()
    bad_rows = []

    for split in ['train', 'valid', 'test']:
        img_dir = dataset_root / split / 'images'
        img_files = sorted([p for p in img_dir.glob('*') if p.suffix.lower() in IMAGE_EXTS]) if img_dir.exists() else []
        for img_path in img_files:
            src = infer_source_from_name(img_path.name)
            source_counts[(split, src)] += 1
            lbl_path = label_file_for_image(img_path)
            lines = lbl_path.read_text().splitlines() if lbl_path.exists() else []
            seen = set()
            valid = 0
            for line_no, line in enumerate(lines, start=1):
                if not line.strip():
                    continue
                parts = line.split()
                if len(parts) != 5:
                    bad_rows.append({'split': split, 'file': img_path.name, 'line': line_no, 'reason': 'bad_column_count', 'raw': line})
                    continue
                try:
                    cls_id = int(float(parts[0]))
                    x, y, w, h = [float(v) for v in parts[1:]]
                except Exception:
                    bad_rows.append({'split': split, 'file': img_path.name, 'line': line_no, 'reason': 'parse_error', 'raw': line})
                    continue
                if cls_id < 0 or cls_id >= len(names):
                    bad_rows.append({'split': split, 'file': img_path.name, 'line': line_no, 'reason': 'bad_class_id', 'raw': line})
                    continue
                if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                    bad_rows.append({'split': split, 'file': img_path.name, 'line': line_no, 'reason': 'bad_coords', 'raw': line})
                    continue
                cls = names[cls_id]
                box_counts[(split, cls)] += 1
                seen.add(cls)
                valid += 1
            if valid == 0:
                empty_counts[split] += 1
            for cls in seen:
                image_counts[(split, cls)] += 1
        split_rows.append({'split': split, 'images': len(img_files), 'empty_label_images': empty_counts[split]})

    for cls in names:
        row = {'class': cls}
        for split in ['train', 'valid', 'test']:
            row[f'{split}_images'] = image_counts[(split, cls)]
            row[f'{split}_boxes'] = box_counts[(split, cls)]
        row['total_images'] = sum(row[f'{s}_images'] for s in ['train', 'valid', 'test'])
        row['total_boxes'] = sum(row[f'{s}_boxes'] for s in ['train', 'valid', 'test'])
        row['target_images'] = TARGET_IMAGES.get(cls, 1000)
        row['still_need'] = max(0, row['target_images'] - row['total_images'])
        class_rows.append(row)

    split_df = pd.DataFrame(split_rows)
    class_df = pd.DataFrame(class_rows).sort_values(['still_need', 'total_images'], ascending=[False, True])
    source_df = pd.DataFrame([
        {'split': split, 'source': source, 'images': count}
        for (split, source), count in source_counts.items()
    ]).sort_values(['split', 'source'])
    bad_df = pd.DataFrame(bad_rows)

    split_df.to_csv(WORK_META / 'latest_split_summary.csv', index=False)
    class_df.to_csv(WORK_META / 'latest_class_summary.csv', index=False)
    source_df.to_csv(WORK_META / 'latest_source_summary.csv', index=False)
    bad_df.to_csv(WORK_META / 'latest_bad_labels.csv', index=False)
    return split_df, class_df, source_df, bad_df

split_summary, class_summary, source_summary, bad_labels_df = audit_dataset()
display(split_summary)
display(source_summary)
display(class_summary)
print('Bad label rows:', len(bad_labels_df))
print('Missing classes:', class_summary[class_summary.total_images == 0]['class'].tolist())
print('Classes below target:', int((class_summary.still_need > 0).sum()))


In [ ]:
# Cell 7: Optional Trial 7 performance context
# This merges Trial 7 corrected test GALVAT scores if you uploaded/copied them to Drive.

TRIAL7_PERF_CANDIDATES = [
    Path('/content/drive/MyDrive/galvat_trial7_final_model_outputs/galvat_trial7_yolo26s_clean640_2026-06-12_15-37-27/training_run/selected_threshold_test_eval_corrected/test_selected_supported_galvat_scores.csv'),
    Path('/content/drive/MyDrive/galvat_trial7_training_runs/galvat_trial7_yolo26s_clean640_2026-06-12_15-37-27/selected_threshold_test_eval_corrected/test_selected_supported_galvat_scores.csv'),
]

perf_df = pd.DataFrame()
for p in TRIAL7_PERF_CANDIDATES:
    if p.exists():
        perf_df = pd.read_csv(p)
        print('Loaded Trial 7 corrected performance:', p)
        break

if len(perf_df):
    keep_cols = [c for c in ['class', 'precision', 'recall', 'map50', 'map50_95', 'class_galvat_score'] if c in perf_df.columns]
    view = class_summary.merge(perf_df[keep_cols], on='class', how='left')
else:
    print('No Trial 7 corrected performance CSV found in Drive. Counts audit still works.')
    view = class_summary.copy()

view['dataset_status'] = np.where(view.total_images == 0, 'missing', np.where(view.still_need > 0, 'needs_more', 'at_or_above_target'))
view.to_csv(WORK_META / 'latest_counts_with_trial7_performance.csv', index=False)
display(view.sort_values(['dataset_status', 'still_need', 'total_images'], ascending=[True, False, True]))


In [ ]:
# Cell 8: Visual counts dashboard

CHART_DIR = WORK_META / 'charts'
CHART_DIR.mkdir(exist_ok=True)
sns.set_theme(style='whitegrid')

plot_df = class_summary.sort_values('total_images', ascending=True).copy()
plt.figure(figsize=(11, max(8, len(plot_df) * 0.22)))
sns.barplot(data=plot_df, y='class', x='total_images', color='steelblue', label='current')
for pos, (_, r) in enumerate(plot_df.iterrows()):
    plt.plot([r['target_images'], r['target_images']], [pos - 0.35, pos + 0.35], color='firebrick', linewidth=2)
plt.title('Trial 8 Roboflow-only Dataset: Current Images by Class')
plt.xlabel('images containing class')
plt.ylabel('')
plt.tight_layout()
plt.savefig(CHART_DIR / 'class_counts_current_vs_target.png', dpi=180)
plt.show()

need_df = class_summary[class_summary.still_need > 0].sort_values('still_need', ascending=False)
display(need_df[['class', 'total_images', 'target_images', 'still_need']])
need_df[['class', 'total_images', 'target_images', 'still_need']].to_csv(WORK_META / 'classes_still_needed.csv', index=False)


In [ ]:
# Cell 9: Add Roboflow sources here
#
# Paste either:
# - the raw Roboflow URL: https://app.roboflow.com/ds/...
# - OR the whole curl command: curl -L "https://app.roboflow.com/ds/..." > roboflow.zip; unzip ...
#
# Set enabled=True for sources you want to import.
# For bathroom signs, use mens_bathroom_sign or womens_bathroom_sign specifically.

ROBOFLOW_SOURCES = [
    {
        'enabled': False,
        'source_name': 'mens_bathroom_sign',
        'target_class': 'mens_bathroom_sign',
        'url_or_script': '',
        'aliases': ['mens_bathroom_sign', 'men bathroom sign', "men's bathroom sign", 'male restroom sign', 'mens restroom sign'],
        'target_total_images': TARGET_IMAGES['mens_bathroom_sign'],
    },
    {
        'enabled': False,
        'source_name': 'womens_bathroom_sign',
        'target_class': 'womens_bathroom_sign',
        'url_or_script': '',
        'aliases': ['womens_bathroom_sign', 'women bathroom sign', "women's bathroom sign", 'female restroom sign', 'womens restroom sign'],
        'target_total_images': TARGET_IMAGES['womens_bathroom_sign'],
    },
    {
        'enabled': False,
        'source_name': 'paper_towel_dispenser',
        'target_class': 'paper_towel_dispenser',
        'url_or_script': '',
        'aliases': ['paper_towel_dispenser', 'paper towel dispenser'],
        'target_total_images': TARGET_IMAGES['paper_towel_dispenser'],
    },
    {
        'enabled': False,
        'source_name': 'hearing_aid',
        'target_class': 'hearing_aid',
        'url_or_script': '',
        'aliases': ['hearing_aid', 'hearing aid'],
        'target_total_images': TARGET_IMAGES['hearing_aid'],
    },
    {
        'enabled': False,
        'source_name': 'tupperware',
        'target_class': 'tupperware',
        'url_or_script': '',
        'aliases': ['tupperware', 'food container', 'storage container', 'plastic container'],
        'target_total_images': TARGET_IMAGES['tupperware'],
    },
]

enabled_sources = [s for s in ROBOFLOW_SOURCES if s.get('enabled') and s.get('url_or_script')]
for s in enabled_sources:
    assert s['target_class'] in final_id, f"Unknown target_class: {s['target_class']}"
print('Enabled Roboflow sources:', len(enabled_sources))
for s in enabled_sources:
    print('-', s['source_name'], '->', s['target_class'])


In [ ]:
# Cell 10: Download and inspect enabled Roboflow datasets

def norm_name(x):
    return re.sub(r'[^a-z0-9]+', ' ', str(x).lower()).strip()

def extract_roboflow_url(url_or_script):
    text = str(url_or_script).strip()
    m = re.search(r'https://app\.roboflow\.com/ds/[^\s"\']+', text)
    if m:
        return m.group(0)
    if text.startswith('http'):
        return text
    raise ValueError(f'Could not find Roboflow download URL in: {text[:120]}')

def download_roboflow_source(src, overwrite=False):
    out_dir = ROBOSRC_ROOT / src['source_name']
    zip_path = ROBOSRC_ROOT / f"{src['source_name']}.zip"
    if overwrite and out_dir.exists():
        shutil.rmtree(out_dir)
    if out_dir.exists() and list(out_dir.rglob('data.yaml')):
        print('Already downloaded:', src['source_name'])
        return out_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    url = extract_roboflow_url(src['url_or_script'])
    print('Downloading', src['source_name'], 'from', url)
    with requests.get(url, stream=True, timeout=180) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        with open(zip_path, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True, desc=src['source_name']) as bar:
            for chunk in r.iter_content(1024 * 1024):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(out_dir)
    zip_path.unlink(missing_ok=True)
    return out_dir

def find_data_yaml(source_dir):
    candidates = list(Path(source_dir).rglob('data.yaml'))
    if not candidates:
        raise FileNotFoundError(f'No data.yaml found in {source_dir}')
    return sorted(candidates, key=lambda p: len(p.parts))[0]

for src in enabled_sources:
    download_roboflow_source(src, overwrite=False)

inspect_rows = []
for src in enabled_sources:
    data_yaml_path = find_data_yaml(ROBOSRC_ROOT / src['source_name'])
    data, names = load_yolo_names(data_yaml_path)
    alias_norms = {norm_name(a) for a in src['aliases']}
    mapped_ids = [i for i, n in enumerate(names) if norm_name(n) in alias_norms]
    single_fallback = len(names) == 1 and not mapped_ids
    split_counts = {}
    label_counts = Counter()
    for split in ['train', 'valid', 'test']:
        img_dir = data_yaml_path.parent / split / 'images'
        lbl_dir = data_yaml_path.parent / split / 'labels'
        imgs = [p for p in img_dir.glob('*') if p.suffix.lower() in IMAGE_EXTS] if img_dir.exists() else []
        split_counts[split] = len(imgs)
        if lbl_dir.exists():
            for lbl in lbl_dir.glob('*.txt'):
                for line in lbl.read_text().splitlines():
                    if line.strip():
                        cls_id = int(float(line.split()[0]))
                        label_counts[names[cls_id] if 0 <= cls_id < len(names) else f'BAD_ID_{cls_id}'] += 1
    inspect_rows.append({
        'source_name': src['source_name'],
        'target_class': src['target_class'],
        'data_yaml': str(data_yaml_path),
        'source_classes': names,
        'mapped_source_class_ids': mapped_ids,
        'single_class_fallback_would_be_used': single_fallback,
        'train_images': split_counts.get('train', 0),
        'valid_images': split_counts.get('valid', 0),
        'test_images': split_counts.get('test', 0),
        'source_label_counts': dict(label_counts),
    })

inspect_df = pd.DataFrame(inspect_rows)
inspect_df.to_csv(WORK_META / 'roboflow_source_inspection.csv', index=False)
display(inspect_df)


In [ ]:
# Cell 11: Quality filters and Roboflow import

SMALL_CLASSES = {
    'keys', 'medicine_bottle', 'electrical_outlet', 'light_switch',
    'toothpaste', 'deodorant', 'razor', 'hearing_aid', 'wallet',
    'thermostat', 'salt_pepper_shaker', 'measuring_cup',
}
THIN_CLASSES = {'cane', 'railing', 'keys', 'writing_device', 'step_curb'}

def current_class_image_counts():
    counts = Counter()
    for split in ['train', 'valid', 'test']:
        lbl_dir = WORK_DS / split / 'labels'
        if not lbl_dir.exists():
            continue
        for lbl_path in lbl_dir.glob('*.txt'):
            seen = set()
            for line in lbl_path.read_text().splitlines():
                if line.strip():
                    cls_id = int(float(line.split()[0]))
                    if 0 <= cls_id < len(FINAL_NAMES):
                        seen.add(FINAL_NAMES[cls_id])
            for cls in seen:
                counts[cls] += 1
    return counts

def split_for_source(source_name, file_name):
    h = int(hashlib.md5(f'trial8-{source_name}-{file_name}'.encode()).hexdigest(), 16) % 100
    if h < 80:
        return 'train'
    if h < 90:
        return 'valid'
    return 'test'

def image_quality_ok(img_path):
    try:
        im = ImageOps.exif_transpose(Image.open(img_path)).convert('L')
        arr = np.asarray(im.resize((256, 256)))
        mean = float(arr.mean())
        std = float(arr.std())
        sharpness = float(
            np.abs(np.diff(arr.astype(np.float32), axis=1)).mean()
            + np.abs(np.diff(arr.astype(np.float32), axis=0)).mean()
        )
        reasons = []
        if mean < 28 or mean > 228:
            reasons.append('bad_brightness')
        if std < 16:
            reasons.append('low_contrast')
        if sharpness < 5.0:
            reasons.append('likely_blurry')
        if img_path.stat().st_size < 7000:
            reasons.append('tiny_file')
        return len(reasons) == 0, {'brightness': mean, 'contrast': std, 'sharpness': sharpness, 'reject_reason': ','.join(reasons)}
    except Exception as e:
        return False, {'brightness': None, 'contrast': None, 'sharpness': None, 'reject_reason': f'decode_error:{e}'}

def parse_source_labels(lbl_path, names, mapped_ids, target_class, allow_single_class_fallback):
    raw_lines = lbl_path.read_text().splitlines() if lbl_path.exists() else []
    if not raw_lines:
        return [], ['empty_label']
    out = []
    reasons = []
    for line in raw_lines:
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) != 5:
            reasons.append('bad_column_count')
            continue
        try:
            src_cls = int(float(parts[0]))
            x, y, w, h = [float(v) for v in parts[1:]]
        except Exception:
            reasons.append('parse_error')
            continue
        if src_cls < 0 or src_cls >= len(names):
            reasons.append('bad_class_id')
            continue
        is_target = src_cls in mapped_ids or (allow_single_class_fallback and len(names) == 1)
        if not is_target:
            reasons.append(f'non_target_source_class:{names[src_cls]}')
            continue
        vals = [x, y, w, h]
        if not all(math.isfinite(v) for v in vals):
            reasons.append('nonfinite_coord')
            continue
        if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
            reasons.append('coord_out_of_range')
            continue
        area = w * h
        aspect = max(w / max(h, 1e-6), h / max(w, 1e-6))
        min_area = 0.00012 if target_class in SMALL_CLASSES else 0.0005
        max_aspect = 28 if target_class in THIN_CLASSES else 16
        if area < min_area:
            reasons.append('box_too_small')
            continue
        if area > 0.90:
            reasons.append('box_too_large')
            continue
        if aspect > max_aspect:
            reasons.append('box_aspect_extreme')
            continue
        out.append(f'{final_id[target_class]} {x:.6f} {y:.6f} {w:.6f} {h:.6f}')
    if not out and not reasons:
        reasons.append('no_usable_target_label')
    return out, reasons

current_counts = current_class_image_counts()
selected_rows, rejected_rows, summary_rows = [], [], []

for src in enabled_sources:
    source_name = src['source_name']
    target_class = src['target_class']
    needed = max(0, int(src['target_total_images']) - current_counts.get(target_class, 0))
    if needed <= 0:
        print(f'Skipping {source_name}: {target_class} already at target.')
        continue

    data_yaml_path = find_data_yaml(ROBOSRC_ROOT / source_name)
    data, names = load_yolo_names(data_yaml_path)
    alias_norms = {norm_name(a) for a in src['aliases']}
    mapped_ids = [i for i, n in enumerate(names) if norm_name(n) in alias_norms]
    allow_single = len(names) == 1 and not mapped_ids
    candidates = []

    for original_split in ['train', 'valid', 'test']:
        img_dir = data_yaml_path.parent / original_split / 'images'
        lbl_dir = data_yaml_path.parent / original_split / 'labels'
        if not img_dir.exists():
            continue
        for img_path in img_dir.iterdir():
            if img_path.suffix.lower() not in IMAGE_EXTS:
                continue
            lbl_path = lbl_dir / f'{img_path.stem}.txt'
            lines_out, label_reasons = parse_source_labels(lbl_path, names, mapped_ids, target_class, allow_single)
            if not lines_out:
                rejected_rows.append({'source_name': source_name, 'target_class': target_class, 'file_name': img_path.name, 'reason': ';'.join(sorted(set(label_reasons)))})
                continue
            ok, q = image_quality_ok(img_path)
            if not ok:
                rejected_rows.append({'source_name': source_name, 'target_class': target_class, 'file_name': img_path.name, 'reason': q['reject_reason']})
                continue
            label_count = len(lines_out)
            if label_count > 16:
                rejected_rows.append({'source_name': source_name, 'target_class': target_class, 'file_name': img_path.name, 'reason': f'too_many_labels:{label_count}'})
                continue
            score = min(q['contrast'], 90) * 0.25 + min(q['sharpness'], 45) * 0.75 + (8 if 1 <= label_count <= 5 else -4)
            candidates.append({'img_path': img_path, 'source_name': source_name, 'target_class': target_class, 'original_split': original_split, 'label_lines': lines_out, 'label_count': label_count, 'score': score, **q})

    candidates = sorted(candidates, key=lambda r: r['score'], reverse=True)
    picked = candidates[:needed]

    for row in tqdm(picked, desc=f'Importing {source_name}->{target_class}'):
        img_path = row['img_path']
        out_split = split_for_source(source_name, img_path.name)
        out_stem = f"rf8_{source_name}_{hashlib.md5(str(img_path).encode()).hexdigest()[:10]}_{img_path.stem}"
        out_img = WORK_DS / out_split / 'images' / f'{out_stem}.jpg'
        out_lbl = WORK_DS / out_split / 'labels' / f'{out_stem}.txt'

        im = ImageOps.exif_transpose(Image.open(img_path)).convert('RGB')
        im = im.resize((640, 640), Image.Resampling.LANCZOS)
        im.save(out_img, quality=92)
        out_lbl.write_text('\n'.join(row['label_lines']) + '\n')
        selected_rows.append({'source_name': source_name, 'target_class': target_class, 'split': out_split, 'original_split': row['original_split'], 'file_name': out_img.name, 'label_count': row['label_count'], 'score': row['score'], 'brightness': row['brightness'], 'contrast': row['contrast'], 'sharpness': row['sharpness']})

    current_counts[target_class] += len(picked)
    summary_rows.append({'source_name': source_name, 'target_class': target_class, 'needed_before_import': needed, 'accepted_candidates': len(candidates), 'selected': len(picked), 'target_total_images': src['target_total_images'], 'total_after_import_estimate': current_counts[target_class], 'source_classes': names, 'single_class_fallback_used': allow_single})

selected_df = pd.DataFrame(selected_rows)
rejected_df = pd.DataFrame(rejected_rows)
source_summary_df = pd.DataFrame(summary_rows)
selected_df.to_csv(WORK_META / 'roboflow_selected_import_manifest.csv', index=False)
rejected_df.to_csv(WORK_META / 'roboflow_rejected_import_manifest.csv', index=False)
source_summary_df.to_csv(WORK_META / 'roboflow_import_summary.csv', index=False)

print('Selected images:', len(selected_df))
display(source_summary_df)
print('Top rejection reasons:')
display(rejected_df['reason'].value_counts().head(30) if len(rejected_df) else pd.Series(dtype=int))


In [ ]:
# Cell 12: Re-audit after Roboflow import and write missing-class report

split_summary, class_summary, source_summary, bad_labels_df = audit_dataset()
missing = class_summary[class_summary.total_images == 0].copy()
needs = class_summary[class_summary.still_need > 0].sort_values('still_need', ascending=False).copy()

needed_report = {
    'created_at': STAMP,
    'class_count': len(FINAL_NAMES),
    'source_policy': 'COCO and Open Images removed; Roboflow retained/imported',
    'primary_metric': 'GALVAT weighted score',
    'ontology_change': 'bathroom_sign split into mens_bathroom_sign and womens_bathroom_sign',
    'missing_classes': missing['class'].tolist(),
    'classes_below_target': needs[['class', 'total_images', 'target_images', 'still_need']].to_dict(orient='records'),
}
(WORK_META / 'latest_needed_report.json').write_text(json.dumps(needed_report, indent=2))

display(split_summary)
display(source_summary)
display(needs[['class', 'total_images', 'target_images', 'still_need', 'total_boxes']])
print('Missing classes:', missing['class'].tolist())
print('Bad label rows:', len(bad_labels_df))
print('Saved needed report:', WORK_META / 'latest_needed_report.json')


In [ ]:
# Cell 13: Zip checkpoint to Drive
# This is the artifact to continue from if Colab disconnects.

ZIP_LOCAL = Path(f'/content/galvat_trial8_roboflow_only_checkpoint_{STAMP}.zip')
ZIP_DRIVE = DRIVE_OUT / ZIP_LOCAL.name

checkpoint_summary = {
    'created_at': STAMP,
    'source_zip': str(SOURCE_ZIP),
    'dataset_root': str(WORK_DS),
    'metadata_root': str(WORK_META),
    'class_count': len(FINAL_NAMES),
    'source_policy': 'Removed Microsoft COCO and Open Images; retained/imported Roboflow only.',
    'primary_metric': 'GALVAT weighted score',
    'notes': [
        'Generic bathroom_sign is not used in Trial 8.',
        'Use mens_bathroom_sign and womens_bathroom_sign as separate labels.',
        'This checkpoint is dataset-building state, not a trained model.',
    ],
}
(WORK_META / 'trial8_checkpoint_summary.json').write_text(json.dumps(checkpoint_summary, indent=2))

if ZIP_LOCAL.exists():
    ZIP_LOCAL.unlink()

INTERNAL_ROOT = Path('galvat_trial8_roboflow_only_working')
with zipfile.ZipFile(ZIP_LOCAL, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as z:
    for path in tqdm(list(WORK_ROOT.rglob('*')), desc='Zipping Trial 8 checkpoint'):
        if path.is_file():
            z.write(path, INTERNAL_ROOT / path.relative_to(WORK_ROOT))

print('Local zip:', ZIP_LOCAL)
print('Zip size GB:', ZIP_LOCAL.stat().st_size / 1e9)
print('Copying to Drive:', ZIP_DRIVE)
shutil.copy2(ZIP_LOCAL, ZIP_DRIVE)
print('Saved:', ZIP_DRIVE)
